### <span style="color:yellow"> Statistical Summary of the Dataset </span>

In [ ]:
import pandas as pd

# Read the CSV
dataset = pd.read_csv("01_Churn_Modelling.csv")
dataset.head()

In [ ]:
dataset.describe()

In [ ]:
# checking datatypes and null values
dataset.info()

### <span style="color:lightgreen"> Dropping Irrelevant Feature </span>
RowNumber, CustomerId and Surname are irrelivant, so we drop those features.

In [ ]:
dataset.drop(["RowNumber","CustomerId","Surname"], axis=1, inplace=True)
dataset

In [ ]:
# checking datatypes and null values
dataset.info()

### <span style="color:lightblue"> Categorize Geography  </span>
Categorize using One hot Encoding

In [ ]:
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geography = OneHotEncoder()
geography_encoding = onehot_encoder_geography.fit_transform(dataset[['Geography']]).toarray()
geography_encoding


In [ ]:
onehot_encoder_geography.get_feature_names_out(['Geography'])

In [ ]:
# Ingest in DataFrames
geography_encoding_dataFrames = pd.DataFrame(geography_encoding, columns=onehot_encoder_geography.get_feature_names_out(['Geography']))
geography_encoding_dataFrames

In [ ]:
## Combine one hot encoder columns with the original data
dataset = pd.concat((dataset.drop('Geography',axis=1), geography_encoding_dataFrames),axis=1)
dataset.head()

In [ ]:
# checking datatypes and null values
dataset.info()

### <span style="color:orange"> Categorize Gender  </span>

In [ ]:
from sklearn.preprocessing import LabelEncoder
l_encoder_gender = LabelEncoder()
dataset['Gender'] =  l_encoder_gender.fit_transform(dataset['Gender'])
dataset

In [ ]:
# checking datatypes and null values
dataset.info()

### <span style="color:green"> Devide Data into Features  </span>
Dependent Feature - Exited
InDependent Feature - Rest of Features

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ind_feature = dataset.drop('Exited',axis=1)
d_feature = dataset['Exited']

# Split the data in training and tetsing sets
x_train,x_test,y_train,y_test = train_test_split(ind_feature, d_feature, test_size=0.33, random_state=42)

# Scale these features
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

x_train
x_test

In [ ]:
import pickle

with open("dataScaler.pkl", "wb") as file:
    pickle.dump(scaler, file)

In [ ]:
with open("gender_encoder.pkl", "wb") as file:
    pickle.dump(l_encoder_gender, file)

with open("geography_encoder.pkl", "wb") as file:
    pickle.dump(onehot_encoder_geography, file)

In [ ]:
dataset

### <span style="color:Red"> ANN Implementation  </span>

In [ ]:
x_train.shape[1]

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Build Our ANN Model
model = Sequential([
    Dense(128, activation='relu', input_shape=(x_train.shape[1],)), # HL1
    Dense(64, activation='relu'), # HL2
    Dense(1, activation='sigmoid') # output L
])

model.summary()


In [ ]:
# Train Model for Binary Clasification
optimize = tf.keras.optimizers.Adam(learning_rate=0.01)
loss = tf.keras.losses.BinaryCrossentropy()

# Compile Model
model.compile(optimizer=optimize, loss=loss, metrics=['accuracy'])

Key Functions of an Optimizer

Weight Adjustment: Updates the model's weights (parameters) iteratively based on the loss function's gradient with respect to the weights.

Minimizing Loss: The optimizer reduces the error (or loss) between the predicted and actual outputs.

Convergence: Ensures the training process converges to the best possible weights that make the model as accurate as possible.

### <span style="color:Skyblue"> SetUp TensorBoard for Monitoring (Getting Stuck) </span>


In [ ]:
# Setting up TensorBoard

from tensorflow.keras.callbacks import TensorBoard, EarlyStopping

log_dir = "logs/modelProcessing"
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

# Setting up Early Stopping
early_stopping_callback = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

# Train the model
model_training = model.fit(
    x_train, y_train, validation_data=(x_test, y_test), epochs=100, 
    callbacks=[tensorflow_callback,early_stopping_callback]
)

In [ ]:
model.save("trained_model.h5")

In [4]:
# Load Tensorboard Extension
%load_ext tensorboard


The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [5]:
%tensorboard --logdir logs/

Reusing TensorBoard on port 6006 (pid 57461), started 0:00:51 ago. (Use '!kill 57461' to kill it.)